In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s — %(levelname)s — %(message)s',
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import subprocess
import sys
import time as _time
from pathlib import Path
from scipy.spatial.distance import cdist
import xarray as xr

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

from pyS3M.simulation.multicolour import sample_chip_patches_batch, sanitise_calibration_maps

# Shared with the per-task worker so notebook and worker can't drift apart
from _bayer_noise_sweep_task import (
    CHIP_SAMPLING_SEED,
    DEFAULT_CALIB_DIR,
    NOISE_MODELS,
    load_calibration,
    photon_space,
)

FAST_MODE = True  # False -> the published 200-photon-level x 100,000-bootstrap sweep
print(f"FAST_MODE={FAST_MODE}")

# Raw results (HDF5, ~3 GB per task in full mode) go to the lab backup drive
RAW_ROOT = Path("/media/u2473660/JSBLeeLabBackup/Data/20261002_BayesianComparison")
FIGURE_DIR = Path("figures")  # git-ignored (*.svg)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sweep_tag = "fast" if FAST_MODE else "full"
save_folder = RAW_ROOT / sweep_tag
save_folder.mkdir(parents=True, exist_ok=True)
print(f"Raw results will be saved to: {save_folder}")

# FigureSI_ — Bayer: median camera noise vs per-bootstrap real-chip noise

Every other SI precision sweep gives each simulated pixel the **sensor-wide median**
gain/offset/variance/rqe of the Ximea chip, so hot or defective pixels never show up in a
simulated precision estimate. This notebook asks how much that matters. Only the standard
Bayer (RGGB) pattern is simulated, with two camera-noise models:

| Noise model | Calibration used for each bootstrap frame |
|---|---|
| **median** | Flat 14×14 tile at the chip-wide median (the usual approach) |
| **chip** | Its own randomly located 14×14 crop of the real full-chip gain/offset/variance/rqe maps |

In the **chip** model, each bootstrap sample *k* gets one crop, drawn once and reused at every
photon level, just as its ground-truth position is. The same crop is used to generate the
frame and to convert and weight it for fitting, so the fit sees exactly the per-pixel
calibration (and per-pixel read noise, $\sqrt{\mathrm{variance}}/\mathrm{gain}$) that the
real pipeline would use for a punctum on that patch of sensor. Crop offsets are saved per
dye (`*_chipcrop_offsets.csv`), so any bootstrap can be traced back to its sensor location.
The 3 physically invalid chip pixels (gain or rqe ≤ 0) are replaced by the chip median.
Hot pixels are kept.

Dyes: **ATTO 488**, **ATTO 565**, **ATTO 647N**. Photon levels run from 200 to 50,000,
log-spaced.

Each (noise model, dye) task runs as its own subprocess (`_bayer_noise_sweep_task.py`).

**`FAST_MODE` flag (set above):** `True` (default) runs 10 photon levels × 2,000 bootstraps,
which takes a few minutes. `False` reproduces the full sweep (200 photon levels × 100,000
bootstraps per noise model and dye), which takes hours of compute. Raw results are written
under `RAW_ROOT/{fast,full}/`.

In [ ]:
# ── Real Ximea calibration: how often are crops affected? ─────────────────────
calib = load_calibration(DEFAULT_CALIB_DIR)
chip_maps, n_invalid = sanitise_calibration_maps(
    {k: calib[k] for k in ("gain", "offset", "variance", "rqe")}
)
image_size = 14  # Bayer 2x2 unit cell x n_unit_cells=7, as in the sweep

HOT_VARIANCE = 10.0  # ADU^2; ~10x the chip median -- what we call a "hot" pixel below
var_med = float(np.median(chip_maps["variance"]))
hot_frac = float(np.mean(chip_maps["variance"] > HOT_VARIANCE))
print(f"chip shape {chip_maps['gain'].shape}, invalid pixels replaced: {n_invalid}")
print(f"median variance {var_med:.3f} ADU^2, max {chip_maps['variance'].max():.0f} ADU^2")
print(f"pixels with variance > {HOT_VARIANCE:g}: {hot_frac:.2e} of chip")
print(f"-> P(a {image_size}x{image_size} crop contains one) ≈ "
      f"{1 - (1 - hot_frac) ** (image_size ** 2):.3f}  (independent-pixel estimate)")

In [ ]:
# ── Example crops: variance maps for a few bootstrap frames ───────────────────
# Uses the worker's seed, so these are frames 0..n-1 of the real sweep's crop draw.
n_show = 6
crops, r0, c0 = sample_chip_patches_batch(
    chip_maps, (image_size, image_size), 2000, np.random.default_rng(CHIP_SAMPLING_SEED)
)
crop_max_var = crops["variance"].reshape(len(r0), -1).max(axis=1)
hot_idx = np.flatnonzero(crop_max_var > HOT_VARIANCE)
show = list(np.flatnonzero(crop_max_var <= HOT_VARIANCE)[: n_show - 2]) + list(hot_idx[:2])

fig, axs = plt.subplots(1, n_show, figsize=(6.69, 1.4))
vmax = np.log10(max(crop_max_var[show]))
for ax, k in zip(axs, show):
    im = ax.imshow(np.log10(crops["variance"][k]), cmap="magma",
                   vmin=np.log10(var_med) - 0.3, vmax=vmax, origin="lower")
    ax.set_title(f"frame {k}\n({r0[k]}, {c0[k]})", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, fraction=0.02, label=r"log$_{10}$ variance / ADU$^2$")
plt.savefig(FIGURE_DIR / "bayer_noise_example_crops.svg", dpi=300, format="svg")
plt.show()
print(f"{len(hot_idx)}/{len(r0)} of these crops contain a pixel with variance > {HOT_VARIANCE:g}")

In [ ]:
# ── Sweep configuration ───────────────────────────────────────────────────────
sim_pixel_size = 69
sim_dyes = ["ATTO 488", "ATTO 565", "ATTO 647N"]
noise_models = list(NOISE_MODELS)  # ["median", "chip"]

N_PHOTON_STEPS = 10 if FAST_MODE else 200
N_BOOTSTRAP = 2_000 if FAST_MODE else 100_000

n_photon_space = photon_space(N_PHOTON_STEPS)  # 200 -> 50,000, log-spaced
print(f"Noise models: {noise_models}")
print(f"Dyes: {sim_dyes}")
print(f"Photon levels: {len(n_photon_space)}  ({n_photon_space[0]:.0f}–{n_photon_space[-1]:.0f})")
print(f"Bootstraps per level: {N_BOOTSTRAP:,}")

In [ ]:
# ── Run simulations: one subprocess per (noise model, dye) ────────────────────
sweep_script = os.path.join(os.getcwd(), "_bayer_noise_sweep_task.py")
n_total = len(noise_models) * len(sim_dyes)
i_task = 0
t0 = _time.time()

for noise_model in noise_models:
    for dye in sim_dyes:
        i_task += 1
        print(f"[{i_task}/{n_total}]  {noise_model:7s}  {dye}", end="  ", flush=True)
        result = subprocess.run(
            [
                sys.executable, sweep_script,
                "--noise-model", noise_model,
                "--dye", dye,
                "--save-folder", str(save_folder),
                "--n-bootstrap", str(N_BOOTSTRAP),
                "--n-photon-levels", str(N_PHOTON_STEPS),
            ],
            capture_output=True, text=True,
        )
        elapsed = (_time.time() - t0) / 60
        eta = (n_total - i_task) * elapsed / i_task
        if result.returncode == 0:
            print(f"done  ({elapsed:.1f} min, ETA {eta:.1f} min)")
        else:
            print(f"FAILED  (exit {result.returncode}, {elapsed:.1f} min)")
            print(result.stderr[-2000:])

print(f"\nAll simulations complete in {(_time.time() - t0) / 60:.1f} min.")

In [ ]:
# ── Load results → Bayer_Noise_XR ─────────────────────────────────────────────
# Dims: [noise_model, dye, n_photons, metric]. For the chip model the sigma_xy is
# additionally split by whether that bootstrap's crop contains a hot pixel.
_metrics = ["sigma_xy", "sigma_colour", "sigma_xy_hotcrop", "sigma_xy_cleancrop"]
Bayer_Noise_XR = xr.DataArray(
    np.full([len(noise_models), len(sim_dyes), len(n_photon_space), len(_metrics)], np.nan),
    coords={"noise_model": noise_models, "dye": sim_dyes,
            "n_photons": n_photon_space, "metric": _metrics},
    dims=["noise_model", "dye", "n_photons", "metric"],
)
hot_fraction = {}

def _sigma_xy(ex, ey):
    return np.sqrt((np.nanstd(ex) ** 2 + np.nanstd(ey) ** 2) / 2) * sim_pixel_size

for i_m, noise_model in enumerate(noise_models):
    flag = NOISE_MODELS[noise_model]
    for i_d, dye in enumerate(sim_dyes):
        stem = save_folder / f"{flag}LM_method_{dye.replace('/', '-')}_"
        raw_file = Path(f"{stem}rawresults.h5")
        gt_file = Path(f"{stem}fittesting_input_groundtruthpositions.csv")
        ip_file = Path(f"{stem}fittesting_input_parameters.csv")
        if not (raw_file.exists() and gt_file.exists() and ip_file.exists()):
            print(f"Missing files for {noise_model} / {dye}")
            continue

        gt = pd.read_csv(gt_file)
        gt_px = np.vstack([gt["x0"].to_numpy(), gt["y0"].to_numpy()]).T / sim_pixel_size
        dye_BGR = pd.read_csv(ip_file).to_numpy()[0][-3:]
        colour_true = np.expand_dims(dye_BGR / dye_BGR.sum(), 0)

        hot_crop = None
        if noise_model == "chip":
            off = pd.read_csv(f"{stem}fittesting_chipcrop_offsets.csv")
            crops_v, _, _ = sample_chip_patches_batch(
                {"variance": chip_maps["variance"]}, (image_size, image_size), len(off), None,
                offsets=(off["row0"].to_numpy(), off["col0"].to_numpy()),
            )
            hot_crop = crops_v["variance"].reshape(len(off), -1).max(axis=1) > HOT_VARIANCE
            hot_fraction[dye] = hot_crop.mean()

        df_raw = pd.read_hdf(raw_file)
        for k in range(len(n_photon_space)):
            sub = df_raw[df_raw["photon_level"] == k]
            frame = sub["frame"].to_numpy().astype(int)
            filt = ~(
                (sub["xc"] < 0) | (sub["yc"] < 0) | (sub["s_x"] < 0) | (sub["s_y"] < 0) |
                (sub["xc"] > image_size) | (sub["yc"] > image_size)
            ).to_numpy()
            if filt.sum() < 5:
                continue
            ex = sub["xc"].to_numpy() - gt_px[frame, 0]
            ey = sub["yc"].to_numpy() - gt_px[frame, 1]
            Bayer_Noise_XR[i_m, i_d, k, 0] = _sigma_xy(ex[filt], ey[filt])
            colour = sub[["A_B", "A_G", "A_R"]].to_numpy()[filt]
            Bayer_Noise_XR[i_m, i_d, k, 1] = float(np.nanstd(cdist(colour, colour_true)))
            if hot_crop is not None:
                for j, sel in ((2, hot_crop[frame]), (3, ~hot_crop[frame])):
                    if (filt & sel).sum() >= 5:
                        Bayer_Noise_XR[i_m, i_d, k, j] = _sigma_xy(ex[filt & sel], ey[filt & sel])

Bayer_Noise_XR.to_netcdf(save_folder / "Bayer_Noise_XR.nc")
print(f"Saved  shape={Bayer_Noise_XR.shape}  dims={list(Bayer_Noise_XR.dims)}")
print({d: f"{f:.3f}" for d, f in hot_fraction.items()}, "<- fraction of chip-model crops with a hot pixel")

In [ ]:
# ── σ_xy and σ_colour vs N_photons: median vs chip noise ──────────────────────
Bayer_Noise_XR = xr.load_dataarray(save_folder / "Bayer_Noise_XR.nc")

_styles = {
    "median": dict(color="black",   linewidth=1.5, linestyle="-",  label="median noise"),
    "chip":   dict(color="#e6194b", linewidth=1.2, linestyle="--", label="real-chip noise"),
}
fs = 8
fig, axs = plotter.two_column_plot(ncols=3, nrows=2, width_ratios=[1, 1, 1], height_ratios=[1, 1])

for row, metric, ylabel in [(0, "sigma_xy", r"$\sigma_{xy}$ / nm"),
                            (1, "sigma_colour", r"$\sigma_{\rm colour}$")]:
    for d_num, dye in enumerate(sim_dyes):
        ax = axs[row, d_num]
        for noise_model, style in _styles.items():
            y = Bayer_Noise_XR.sel(noise_model=noise_model, metric=metric, dye=dye)
            ax = plotter.line_plot(ax, n_photon_space, y, **style)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(r"$N_{\rm photons}$", fontsize=fs)
        ax.set_ylabel(ylabel, fontsize=fs)
        ax.set_xlim([n_photon_space[0], n_photon_space[-1]])
        ax.tick_params(labelsize=fs - 1)
        if row == 0:
            ax.set_title(dye, fontsize=fs)
        ax.legend(loc="best", fontsize=fs - 2, framealpha=0.7)

plt.savefig(FIGURE_DIR / "bayer_noise_precision_comparison.svg", dpi=300, format="svg")
plt.show()

In [ ]:
# ── Chip noise: effect of hot pixels, and ratio to the median-noise curve ─────
fig, axs = plotter.two_column_plot(ncols=3, nrows=2, width_ratios=[1, 1, 1], height_ratios=[1, 1])
for d_num, dye in enumerate(sim_dyes):
    ax = axs[0, d_num]
    sel = lambda m, metric: Bayer_Noise_XR.sel(noise_model=m, metric=metric, dye=dye)
    ax = plotter.line_plot(ax, n_photon_space, sel("median", "sigma_xy"),
                           color="black", linewidth=1.5, label="median noise")
    ax = plotter.line_plot(ax, n_photon_space, sel("chip", "sigma_xy_cleancrop"),
                           color="#4363d8", linewidth=1.2, label="chip, no hot pixel")
    ax = plotter.line_plot(ax, n_photon_space, sel("chip", "sigma_xy_hotcrop"),
                           color="#f58231", linewidth=1.2, label="chip, hot pixel in crop")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_ylabel(r"$\sigma_{xy}$ / nm", fontsize=fs)
    ax.set_title(dye, fontsize=fs)
    ax.legend(loc="best", fontsize=fs - 2, framealpha=0.7)

    ax = axs[1, d_num]
    for metric, col, lab in [("sigma_xy", "#e6194b", r"$\sigma_{xy}$"),
                             ("sigma_colour", "#3cb44b", r"$\sigma_{\rm colour}$")]:
        ratio = sel("chip", metric) / sel("median", metric)
        ax = plotter.line_plot(ax, n_photon_space, ratio, color=col, linewidth=1.2, label=lab)
    ax.axhline(1.0, color="grey", lw=0.6, ls=":")
    ax.set_xscale("log")
    ax.set_ylabel("chip / median", fontsize=fs)
    ax.legend(loc="best", fontsize=fs - 2, framealpha=0.7)

for ax in axs.ravel():
    ax.set_xlabel(r"$N_{\rm photons}$", fontsize=fs)
    ax.set_xlim([n_photon_space[0], n_photon_space[-1]])
    ax.tick_params(labelsize=fs - 1)

plt.savefig(FIGURE_DIR / "bayer_noise_hotpixel_split.svg", dpi=300, format="svg")
plt.show()